### Excel Tools

In [2]:
### Import Packages ###
import os, csv, glob
import pandas as pd
import tkinter as tk
from datetime import datetime
from tkinter.filedialog import askopenfilename, askdirectory
print(">>> loaded")

>>> loaded


#### Get Excel Info
Updated 2026-09-26

In [ ]:
def getExcelInfo(folder_directory: str):
    curr_dir = os.getcwd()

    for root, dirs, files in os.walk(curr_dir):
        if folder_directory in root:
            os.chdir(root)
            os.chdir('..')
            log = open(folder_directory + "_excel_info" + ".txt", "a", encoding = 'utf8')
            log.write(f"Log Updated On {datetime.now().strftime("%G%m%d_%H%M")}\n")
            print(f">>> Analysing files from folder >>> [{root}]")
            for file in files:
                if ".csv" in file:
                    df = pd.read_csv(root + "/" + file)
                elif ".xlsx" in file:
                    df = pd.read_excel(root + "/" + file)

                if not df:
                    print(f">> Reading File: '{file}'")
                    columns, rows = len(df.axes[1]), len(df.axes[0])
                    print(f">> # Columns: {columns}")
                    print(f">> # Rows   : {rows}")
                    log.write(f"// {file}\n# Columns: {columns}\n# Rows: {rows}\n\n")
            print(f">>> Updated Log File")
            log.close()
        print(f">>> Could not find '{folder_directory}'")

# Set Folder Directory
folder_directory = "folder_directory" 
getExcelInfo(folder_directory)

#### Record Finder

In [ ]:
curr_dir = os.getcwd()
folder_to_search = "files_to_search"

# 1 - Find specific record in Excel file ###
def findRecords(column_to_search_1, target_value_1, column_to_search_2, target_value_2):
    for root, dirs, files in os.walk(curr_dir):
        if folder_to_search in root:
            for f in files:
                if ".xlsx" in f:
                    input_data = pd.read_excel(root + "\\" + f, dtype = {column_to_search_1: "string", column_to_search_2: "string"})
                elif ".csv" in f:
                    input_data = pd.read_csv(root + "\\" + f, dtype = {column_to_search_1: "string", column_to_search_2: "string"})

                if input_data is not None:
                    print(f"Searching file {f}...")
                    column1 = input_data[column_to_search_1].tolist()
                    column2 = input_data[column_to_search_2].tolist()

                    if any( [(value1 in target_value_1 for value1 in column1), (value2 in target_value_2 for value2 in column2)] ):
                        for i in range(0, len(column1)):
                            if column1[i] in target_value_1:
                                print(f"Target value {column1[i]} found on {column_to_search_1} > row {i+2}")

                            elif column2[i] in target_value_2:
                                print(f"Target value {column2[i]} found on {column_to_search_1} > row {i+2}")

# 2 - For a specific column, identify values that exceed a defined character length
def getLengthOfColumnValues(column_name, targetLength):
    for root, dirs, files in os.walk(curr_dir):
        if folder_to_search in root:
            for f in files:
                if ".xlsx" in f:
                    input_data = pd.read_excel(root + "\\" + f, dtype = {column_to_search_1: "string", column_to_search_2: "string"})
                elif ".csv" in f:
                    input_data = pd.read_csv(root + "\\" + f, dtype = {column_to_search_1: "string", column_to_search_2: "string"})

                print(f"Searching file {f}...")
                column1 = input_data[column_name].tolist()

                for i in range(0, len(column1)):
                    if len(column1[i]) > targetLength:
                        print(f"Row {i+2} > value '{column1[i]}' exceeds {targetLength} characters")

# User defined variables
# 1 - Specify the columns to search from and the values to search for 
column_to_search_1 = "Item Number"
target_value_1 = ["1584C002AA"]

column_to_search_2 = "Country code"
target_value_2 = ["1584C002AA"]

print(f"Finding rows under column {column_to_search_1} with target value {target_value_1} OR under column {column_to_search_2} with target value {target_value_2}.")
findRecords(column_to_search_1, target_value_1, column_to_search_2, target_value_2)

# 2 - Specify the column name and the target character length
column_name = "Item Name"
target_length = 80

print(f"Running function to get length of values of specific column {column_name} that exceed {target_length} characters.")
getLengthOfColumnValues(column_name, target_length)

#### .csv Splitter

In [ ]:
row_limit = 100000 # Define the number of rows per split file

def split_csv(input_file, row_limit, output_prefix):
    with open(input_file, 'r', newline = '', encoding = "utf8") as csvfile:
        reader = csv.reader(csvfile)
        header = next(reader)
        
        file_number = 1
        rows_written = 0
        
        output_file = f"{output_prefix}_{file_number}.csv"
        outfile = open(output_file, 'w', newline = '', encoding = "utf8")
        writer = csv.writer(outfile)
        writer.writerow(header)
        
        for row in reader:
            if rows_written >= row_limit:
                outfile.close()
                file_number += 1
                rows_written = 0
                output_file = f"{output_prefix}_{file_number}.csv"
                outfile = open(output_file, 'w', newline = '', encoding = "utf8")
                writer = csv.writer(outfile)
                writer.writerow(header)
                
            writer.writerow(row)
            rows_written += 1
        
        outfile.close()

        return file_number

filename = ""

while ".csv" not in filename:
    print(">>> Please select the .csv file to be split.")
    filepath = askopenfilename()
    filename = os.path.basename(filepath)
    print(f">>> Selected file: {filename}")

with open(filepath, 'r', newline='', encoding="utf8") as csvfile:
    rows = sum(1 for row in csv.reader(csvfile))

print(f">>> Selected file has {rows} rows.")
print(f">>> Processing {filename} now...")
output_filename = filename.replace(".csv", "")
num_of_files = split_csv(filepath, row_limit, output_filename)

print(f"Finished spltting {filename} into {num_of_files} parts, with maximum of {row_limit} rows each.")

#### Combine Excel Files

In [ ]:
curr_dir = os.getcwd()
# print(f"Current directory: {curr_dir}\n")

# Select directory
def setup():
    print("Select the folder directory containing the Excel files you wish to combine.")
    root = tk.Tk()
    root.withdraw()
    folder_path = filedialog.askdirectory(title='Select Folder Directory')

    # Find .xlsx / .csv files
    if folder_path:
        xlsx_files = glob.glob(os.path.join(folder_path, '*.xlsx'))
        csv_files = glob.glob(os.path.join(folder_path, '*.csv'))

        print(f"Selected the target folder '{folder_path}', which contains:")

        if not xlsx_files and not csv_files:
            print("> No .xlsx or .csv files\n")
        else:
            print(f"> {len(xlsx_files)} .xlsx files")
            print(f"> {len(csv_files)} .csv files\n")

            excel_files = xlsx_files + csv_files
            user_input(excel_files)
    else:
        print("No folder directory was selected.\n")

# User Decisions
def user_input(excel_files):
    # print("Proceed to combine these files? (y/n)")
    # proceed = input(">>> ")

    print("Choose the type of combine.")
    print("> 1. Each file as a separate sheet.")
    print("> 2. All files combined into one sheet.")
    combine_type = str()
    
    while not combine_type:
        combine_type = input(">>> ")
        if combine_type not in ["1", "2"]:
            print("> The program did not understand which combine type to execute.")
            print("> Please enter either 1 or 2.\n")
            combine_type  = ""

    print("\nGive a name for the combined file. Do not specify the file extension.")
    file_name = input(">>> ") + ".xlsx"
    output_path = os.path.join(curr_dir, file_name)

    if combine_type == "1":
        combine_per_sheet(excel_files, output_path)
    elif combine_type == "2":
        combine_one_sheet(excel_files, output_path)

# 1 - Each file - One sheet
def combine_per_sheet(excel_files, output_path):
    with pd.ExcelWriter(output_path) as writer:
        for file in excel_files:
            sheet_name = os.path.basename(file).split('.')[0]
            
            sheet_name = sheet_name[:31] if len(sheet_name) > 31 else sheet_name

            if ".xlsx" in file:
                df = pd.read_excel(file)
            elif ".csv" in file:
                df = pd.read_csv(file)
            
            # Write the DataFrame to a specific sheet in the new Excel file
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"> Added {file} as sheet '{sheet_name}' to the combined file.")

    print(f"\n> Successfully combined the files into '{output_path}'\n")

# 2 - All files - One sheet
def combine_one_sheet(excel_files, output_path):
    df_list = []

    for file in excel_files:
        if ".xlsx" in file:
            df = pd.read_excel(file)
        elif ".csv" in file:
            df = pd.read_csv(file)

        df_list.append(df)
        print(f"> Added file '{file}' to the combined file.")

    merged_df = pd.concat(df_list, ignore_index=True)
    merged_df.to_excel(output_path, index=False)

    print(f"\n> Successfully combined the files into '{output_path}'\n")

setup()


#### Compare Two Excel Files With Datatables

In [ ]:
# Datatable 1
filename1 = ""
while ".xlsx" not in filename1:
    print(">>> Please select the First Excel file.")
    filepath1 = askopenfilename()
    filename1 = os.path.basename(filepath1)
    print(">>> Selected file 1: ", filename1)

# Datatable 2
filename2 = ""
while ".xlsx" not in filename2:
    print(">>> Please select the Second Excel file.")
    filepath2 = askopenfilename()
    filename2 = os.path.basename(filepath2)
    print(">>> Selected file 2: ", filename2)

# Compare datatables
print(f">>> Comparing '{filename1}' and '{filename2}'...")
df1 = pd.read_excel(filepath1)
# print("> file 1's info")
# print(df1.shape)

df2 = pd.read_excel(filepath2)
# print("> file 2's info")
# print(df2.shape)

df1.compare(df2)